# SAM ViT-B · сегментация по одной точке

In [ ]:
import hashlib
import json
import os
import zipfile
from pathlib import Path

project = "sam"
archive_name = "replaceme-sam-env.zip"
expected_manifest = "2e0fb48db4670f2178318e10ecce69a21a54d05f9d716ade867ddce00b4a63ee"
kaggle = Path("/kaggle/input").is_dir()
default_input = Path("/kaggle/input/replaceme-sam-env") if kaggle else Path.cwd()
inputs = Path(os.environ.get("REPLACEME_INPUT", default_input)).expanduser().resolve()
work = Path(os.environ.get("REPLACEME_WORK", "/kaggle/working" if kaggle else "replaceme-work")).resolve()

archive_path = inputs if inputs.name == archive_name else inputs / archive_name
if archive_path.is_file():
    bundle = archive_path
    with zipfile.ZipFile(bundle) as archive:
        metadata = archive.read("environment.json")
        bootstrap = archive.read("bootstrap.py")
elif inputs.is_dir() and (inputs / "environment.json").is_file():
    bundle = inputs
    metadata = (bundle / "environment.json").read_bytes()
    bootstrap = (bundle / "bootstrap.py").read_bytes()
else:
    raise FileNotFoundError(f"Ожидается {archive_path}. Укажите точный путь к ZIP или каталогу Dataset в REPLACEME_INPUT.")

if hashlib.sha256(metadata).hexdigest() != expected_manifest:
    raise ValueError(f"Манифест не соответствует комплекту {archive_name}.")
manifest = json.loads(metadata)
if manifest["project"] != project or hashlib.sha256(bootstrap).hexdigest() != manifest["bootstrap_sha256"]:
    raise ValueError("Ноутбук и загрузчик окружения относятся к разным сборкам.")
scope = {}
exec(compile(bootstrap, "bootstrap.py", "exec"), scope)
scope["start"](bundle, manifest, work / project)


## Загрузка весов и данных

In [ ]:
%%benchmark
"""Prepare the pinned SAM ViT-B checkpoint and 32-image LVIS benchmark."""
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from hashlib import sha256
from http.client import HTTPException
import fcntl
import json
import os
from pathlib import Path
import random
import tempfile
import time
from urllib.parse import urlsplit
from urllib.request import urlopen
from zipfile import ZipFile


WEIGHTS = "sam_vit_b_01ec64.pth"
WEIGHTS_URL = "https://dl.fbaipublicfiles.com/segment_anything/" + WEIGHTS
WEIGHTS_HASH = "ec2df62732614e57411cdcf32a23ffdf28910380d03139ee0f4fcbe91eb8c912"
LVIS_URL = "https://dl.fbaipublicfiles.com/LVIS/lvis_v0.5_val.json.zip"
LVIS_HASH = "bf7bd3a9ff8283991f6ac09c9a70b0a68e6a576e3e9e39588c1e3277ebf6d8ad"
MANIFEST_HASH = "e0a0f20246ae0d569ba2e04c79d1862a2c8f555670fde24426f006583269edfa"
IMAGE_HASHES = {
    5992: 'c33842fbbd9626843b93ede50dd4908221cb0c9bf9837f58861354eec2a4ec56',
    22371: '7023b465285e7fcc25c9272a9b24efc5df076ad982513ff6dec4c3c7842e734e',
    23272: '3a20a6c0f0c7d33b9c4d9a4fe9023446ce3f92e90b37792cb791c8e3d3ac4cb4',
    26465: 'f941bf1156d9fc2831c487a93caf78ae3367ddaa5ba4f6cda4384af13ea91ff0',
    29187: '636a078ae06c6779d11fe5bc889506a6cbf5cad7a3c82dccf1d7d98130bbef73',
    86582: '445975ce033b1845fe2d9f66ea7833bbd0f651be7c621b0b8bc85d741da4a1da',
    93261: 'b21e5e3f4d9c8d16bb42b17092d2f3b73770048148b9782a5ecb99ef34a14272',
    102707: 'a4e68e6ae9eef4884fcc8d010ceaf4f72d55c234dc68a03c027f9aef2976436e',
    112626: '905743b94ff4f909abb524bfb7b17f8d79d89d6bad89f2322d6d9876acbf067a',
    140583: '32fb5c808efcd3187ddbc6449b4a564db0fef52d52c272168a0418835a655cb8',
    156278: '7d3fdff1cdf4fe345c57ea392793a18bf485ab1fd5fdfc53d4d185855f2ae382',
    160666: 'a7bfafb7a35760c7ab6fc4c8afef40f8d79f4ecb6d3659f67bb00f17e2335d88',
    193717: 'a2606516f5b4ff9ba6dd7e65ef87bff0c1621f837ba164954d41c688dc7da16e',
    211825: '9d33bcc07c852c2ed2c87f8873aa101474b21777adac7f4ce0a5590e0af8adba',
    214753: 'cbf50318249d78406eeee4f1e91693acc0f8e05dc9e745632354369caae57777',
    216636: '112a1d4d1a4c142b10bffbec83d07ce906016b72bfd1e6577f6b593b1e8df320',
    220732: '81b9254c750f52ebc8fcc5fa421ab8a3ac430db19f541bfb7541adada76a6a30',
    229216: '627bc997b0d21da6ac5ea5b715a456282e8b5f0f62cd30b6fec45ad1341aaabd',
    240250: 'fb37c81f0e4a91f28ac009ce2ca5316e9070486afa76888548637e141d74b40a',
    273711: 'b146b5b227d5048cf24b027b56d61cd8935c974cba79d7b81e890f317f3aa27d',
    276921: 'e69f971e70247ff09ba6fbdc3af64e12afb87fad359cf15ea0dbbbc841254e85',
    277020: '791eb6de6cf059c67b2b7809cd5132b3708ed39d36a1f67c55714a91391b3cfc',
    334767: '6afc10306fa9c327dd90ac01b1bd79e328fc0583761d1133c8ace1325150788e',
    338718: '0f0085a1167987991d58fd8e34ef0fad20bc5b72030c9d5a972e3eb66a30e9ee',
    417465: '63e271d2cb2f427b518e406b5f1bd838ddf2ec3243fdbc8094bc205d7a3bf739',
    419974: 'eff82e4b2a14a39e1f76d1ef1f4dc986866d985f679367e45c8c071a1f4078cf',
    420916: '49a25255378293de63164929a056228e03feee7c68931c186c9b6ea913742003',
    449190: '4ffc79f36280acc9128ebd27d442816c93faf6ee3008a8a42a085e30c74a72c3',
    503823: '3e88fec88513fd909f489ea1d0be3233025585825672efd85d98e64d9f5be506',
    546659: 'ca49e09b807397a5b5185c451e79d257c92734a447d30e4f86645ab431bf2e1b',
    546826: '4a4089c8b04a04facaaa44cef49aeb2b2897fbaabfa221442ce50e7e67c521ab',
    563349: 'c0cae527104995431e0e90c67a0805089e2ccfeabcc968ac67f0c5290b783792',
}


def _matches(path, expected):
    if not path.is_file():
        return False
    digest = sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(4 << 20), b""):
            digest.update(block)
    return digest.hexdigest() == expected


def _fetch(url, path, expected):
    if _matches(path, expected):
        return
    for attempt in range(3):
        partial = None
        try:
            with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as output:
                partial = Path(output.name)
                digest = sha256()
                with urlopen(url, timeout=90) as response:
                    for block in iter(lambda: response.read(4 << 20), b""):
                        output.write(block)
                        digest.update(block)
            if digest.hexdigest() != expected:
                raise ValueError(f"SHA-256 mismatch for {path.name}")
            os.replace(partial, path)
            return
        except (OSError, ValueError, EOFError, HTTPException) as exc:
            if attempt == 2:
                raise RuntimeError(f"Could not download and verify {url}") from exc
            time.sleep(2 ** attempt)
        finally:
            if partial is not None:
                partial.unlink(missing_ok=True)


def _manifest(directory):
    manifest = directory / "lvis.json"
    if _matches(manifest, MANIFEST_HASH):
        return json.loads(manifest.read_bytes())

    archive = directory / "lvis_v0.5_val.json.zip"
    _fetch(LVIS_URL, archive, LVIS_HASH)
    with ZipFile(archive) as source:
        dataset = json.loads(source.read("lvis_v0.5_val.json"))
    objects = defaultdict(list)
    for annotation in dataset["annotations"]:
        objects[annotation["image_id"]].append(annotation)
    selected = sorted(random.Random(42).sample(sorted(objects), 32))
    metadata = {image["id"]: image for image in dataset["images"]}
    categories = {cat["id"]: cat["name"] for cat in dataset["categories"]}
    images = []
    for image_id in selected:
        image = metadata[image_id]
        url = "https://s3.amazonaws.com/images.cocodataset.org" + urlsplit(image["coco_url"]).path
        annotations = sorted(objects[image_id], key=lambda ann: ann["id"])
        images.append(dict(id=image_id, file=f"{image_id:012}.jpg",
                           height=image["height"], width=image["width"], source=url,
                           flickr_url=image["flickr_url"], license=image["license"],
                           sha256=IMAGE_HASHES[image_id],
                           objects=[dict(id=ann["id"], category=categories[ann["category_id"]],
                                         polygons=ann["segmentation"]) for ann in annotations]))
    result = dict(dataset="LVIS v0.5 val", source=LVIS_URL, seed=42,
                  licenses=dataset["licenses"], images=images)
    data = json.dumps(result, ensure_ascii=False).encode("utf-8")
    if sha256(data).hexdigest() != MANIFEST_HASH:
        raise ValueError("The LVIS sample differs from the reference sample")
    with tempfile.NamedTemporaryFile(dir=directory, delete=False) as out:
        out.write(data)
    os.replace(out.name, manifest)
    return result


def prepare(work):
    """Return a verified asset directory, caching downloads under work."""
    directory = Path(work).resolve() / "sam-lvis-assets"
    images = directory / "images"
    images.mkdir(parents=True, exist_ok=True)
    with (directory / ".prepare.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        manifest = _manifest(directory)
        _fetch(WEIGHTS_URL, directory / WEIGHTS, WEIGHTS_HASH)
        def prepare_image(image):
            _fetch(image["source"], images / image["file"], image["sha256"])
        with ThreadPoolExecutor(max_workers=4) as pool:
            list(pool.map(prepare_image, manifest["images"]))
    return directory


asset_dir = prepare(Path.cwd())

## Модель и выборка

In [ ]:
%%benchmark
import csv
import json
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from pycocotools import mask as coco_mask
from scipy.ndimage import distance_transform_edt
from segment_anything import SamPredictor, sam_model_registry

assets = asset_dir
images = json.loads((assets / "lvis.json").read_text())["images"]
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

model = sam_model_registry["vit_b"]()
model.load_state_dict(torch.load(assets / "sam_vit_b_01ec64.pth", map_location="cpu", weights_only=True))
predictor = SamPredictor(model.eval().to(device))
print(torch.cuda.get_device_name() if device == "cuda" else "CPU", "· PyTorch", torch.__version__)
print(len(images), "изображения ·", sum(len(im["objects"]) for im in images), "объектов")

## Одна точка → три маски

In [ ]:
%%benchmark
def ground_truth(obj, height, width):
    polygons = coco_mask.frPyObjects(obj["polygons"], height, width)
    return coco_mask.decode(coco_mask.merge(polygons)).astype(bool)


def score_object(target):
    distance = distance_transform_edt(np.pad(target, 1))[1:-1, 1:-1]
    y, x = np.unravel_index(distance.argmax(), target.shape)
    assert target[y, x], "Пустая эталонная маска"
    masks, confidence, _ = predictor.predict(
        point_coords=np.array([[x, y]]), point_labels=np.array([1]), multimask_output=True,
    )
    iou = (masks & target).sum(axis=(1, 2)) / (masks | target).sum(axis=(1, 2))
    selected = int(confidence.argmax())
    best = int(iou.argmax())
    row = dict(x=int(x), y=int(y), selected=selected, oracle=best,
               iou=float(iou[selected]), oracle_iou=float(iou[best]))
    return row, masks[selected]


rows = []
with torch.inference_mode():
    for img in images:
        with Image.open(assets / "images" / img["file"]) as photo:
            rgb = np.array(photo.convert("RGB"))
        assert rgb.shape[:2] == (img["height"], img["width"])
        predictor.set_image(rgb)
        for obj in img["objects"]:
            target = ground_truth(obj, *rgb.shape[:2])
            score, predicted = score_object(target)
            if not rows:
                example = (rgb, target, predicted, score)
            rows.append(dict(image_id=img["id"], object_id=obj["id"], category=obj["category"], **score))

## Результат

In [ ]:
%%benchmark
assert len(rows) == sum(len(img["objects"]) for img in images)
assert all(0 <= r["iou"] <= r["oracle_iou"] <= 1 for r in rows)
path = Path.cwd() / "sam_lvis_scores.csv"
with path.open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=rows[0])
    writer.writeheader()
    writer.writerows(rows)

for label, key in [("SAM", "iou"), ("Oracle", "oracle_iou")]:
    print(f'{label:6}  mIoU {np.mean([r[key] for r in rows]):.2%}')
print(path.name, "·", len(rows), "строк")

## Первый объект выборки

In [ ]:
%%benchmark
rgb, target, predicted, score = example
photo = Image.fromarray(rgb)
x, y = score["x"], score["y"]
ImageDraw.Draw(photo).ellipse((x-6, y-6, x+6, y+6), fill="orange", outline="black", width=2)
panels = [photo, Image.fromarray(target).convert("RGB"), Image.fromarray(predicted).convert("RGB")]
for panel in panels:
    panel.thumbnail((320, 250))
w, h = panels[0].size
strip = Image.new("RGB", (3*w + 8, h), "white")
for i, panel in enumerate(panels):
    strip.paste(panel, (i * (w + 4), 0))
print(f'LVIS {rows[0]["image_id"]} · {rows[0]["category"]} · IoU {score["iou"]:.1%}')
display(strip)